# invoice-parser: reading the results

> Any cloud cost file in, the same line items out.

This guide is a notebook: every output below was produced by running the cell above it. It uses
`examples/sample-focus-export.csv`, a small made-up usage export in a format the parser
recognises without a model, so you can run it yourself with no keys and no GPU.

```
uv sync --extra docs
uv run jupyter execute docs/guide.ipynb     # or open it in any notebook editor
```

## Parse a file

`parse` takes a path, bytes or an open file. `use_model=False` is only for this guide; leave it
out for real documents, which need a model (see [Choosing a model](#choosing-a-model)).

In [1]:
from invoice_parser import parse

result = await parse("../examples/sample-focus-export.csv", use_model=False)
result

<ParseResult 1 document(s), pass>

One input can hold several documents (a zip, a workbook with many sheets), so a result is a
list of documents.

In [2]:
doc = result[0]
doc

<ParsedDocument 'sample-focus-export.csv' usage_export, 9 rows, confidence high>

## What the document is

Before looking at any line, a document tells you what it is and how far to trust it.

In [3]:
print("kind:      ", doc.kind)
print("provider:  ", doc.provider)
print("currency:  ", doc.currency)
print("period:    ", doc.period.start.date(), "to", doc.period.end.date())
print("total:     ", doc.total)
print("confidence:", doc.confidence)
print("issues:    ", doc.issues)

kind:       usage_export
provider:   aws
currency:   USD
period:     2026-06-01 to 2026-07-01
total:      None
confidence: high
issues:     []


| Property | Meaning |
|---|---|
| `kind` | `bill`, `usage_export`, `cost_summary`, `estimate`, `infrastructure`, `other`, or `unreadable` (`issues` says why) |
| `total` | The total the document states for itself. `None` here: a usage export states none. |
| `confidence` | `high`, `medium` or `low`, from the parser's own checks (for example, do the line items add up to the stated total). `unknown` when there was nothing to check. |
| `issues` | Readable reasons whenever confidence is not `high` |

## Three lists

Line items come grouped by what you would do with them.

| List | Holds | Use it for |
|---|---|---|
| `doc.usage` | What is being run | The baseline to price |
| `doc.commitments` | Reservations and savings plans already bought | What is already covered, and on what terms |
| `doc.adjustments` | Discounts, credits, tax, fees | What to leave out of the baseline |

`doc.items` is all three together, in document order. Rows that only summarise other rows
(subtotals, recap sections) are never in any of them.

In [4]:
print(
    len(doc.usage),
    "usage,",
    len(doc.commitments),
    "commitment,",
    len(doc.adjustments),
    "adjustments",
)

for item in doc.usage:
    print(f"{item.service.name:36} {item.region.code:10} {str(item.quantity):15} {item.cost}")

5 usage, 1 commitment, 3 adjustments
Amazon Elastic Compute Cloud         us-east-1  2,160 Hours     207.36 USD
Amazon Elastic Compute Cloud         us-east-1  720 Hours       0.00 USD
Amazon Elastic Compute Cloud         us-east-1  500 GB-Months   40.00 USD
Amazon Relational Database Service   us-west-2  720 Hours       123.12 USD
Amazon Simple Storage Service        us-west-2  1,840 GB-Months 42.32 USD


The lists read from disk as they are used, so a million-row export costs nothing until you
iterate it. They support `len()`, indexing, slicing, and a total per currency:

In [5]:
doc.usage.total(), doc.adjustments.total()

({'USD': Decimal('412.8')}, {'USD': Decimal('106.08')})

## One item

Every list holds the same type, `Item`. Its fields are grouped the way a price lookup uses
them.

In [6]:
item = doc.usage[0]

print("kind        ", item.kind)
print("service     ", item.service)
print("region      ", item.region)
print("quantity    ", item.quantity)
print("cost        ", item.cost)
print("unit_price  ", item.unit_price)
print("pricing     ", item.pricing.model)
print("account     ", item.account)
print("description ", item.description)
print("detail      ", item.detail)
print("source      ", item.source.file, "row", item.source.row)

kind         usage
service      Service(name='Amazon Elastic Compute Cloud', code='AmazonEC2', category='Compute')
region       Region(name='US East (N. Virginia)', code='us-east-1', zone=None)
quantity     2,160 Hours
cost         207.36 USD
unit_price   0.096 USD
pricing      on_demand
account      Account(id='444455556666', name='web-prod')
description  $0.096 per On Demand Linux m5.large Instance Hour
detail       sku
source       sample-focus-export.csv row 2


In [7]:
item.product

Product(sku='SKU-EC2-M5L', usage_type='BoxUsage:m5.large', operation=None, family=None, meter=None, instance_type=None, operating_system=None, database_engine=None, deployment=None, storage_type=None, tenancy=None, license=None, extra={})

Two rules hold everywhere:

- **A value the document did not state is `None`.** The parser does not guess. That is why
  `Service` and `Region` carry both a `name` (as the document words it) and a `code` (the
  provider's identifier, present only when the document gives one). Above, `instance_type` is
  `None` because this export has no such column; the machine size is inside `usage_type`.
- **`item.detail` says how precisely the item can be matched** to a price list, best first:

| `detail` | The item has | Match by |
|---|---|---|
| `sku` | A provider SKU | Exact lookup |
| `usage_type` | Service and usage type | Near-exact lookup |
| `attributes` | Service, region and a machine size | Attribute search |
| `description` | Service and a worded description with a price hint | Text matching |
| `service` | Service only | Service-level estimate |
| `none` | None of these | Cannot be priced as is |

## Commitments and adjustments

In [8]:
for item in doc.commitments:
    print(item.description)
    print("  ", item.pricing)
    print("  ", item.cost)

1 year No Upfront Compute Savings Plan, $0.15/hour commitment
   Pricing(model='committed_use', term=None, payment=None, offering_class=None, commitment_id='sp-0123456789abcdef', commitment_type='Spend')
   108.00 USD


In [9]:
for item in doc.adjustments:
    print(f"{item.kind:8} {str(item.cost):14} {item.description}")

credit   -25.00 USD     Promotional credit applied to this invoice
tax      31.08 USD      Sales tax
fee      100.00 USD     AWS Support (Business) monthly fee


`of_kind` narrows a list: `doc.adjustments.of_kind("tax")`.

`pricing.model` reflects what the document says. This export marks the plan and the usage it
covers only as "Committed", so the model reads `committed_use`; a bill that names a Savings
Plan or a Reserved Instance gives `savings_plan` or `reserved`, with `term` and `payment` when
they are stated.

## As a table

For a database or a dataframe, `to_frame()` gives the same fields flat, under the same names.
It returns a [Polars](https://pola.rs) DataFrame.

In [10]:
import polars as pl

pl.Config.set_tbl_width_chars(120)
frame = doc.usage.to_frame()
frame.select("service_code", "region_code", "usage_type", "quantity", "unit", "cost", "detail")

service_code,region_code,usage_type,quantity,unit,cost,detail
str,str,str,"decimal[38,10]",str,"decimal[38,10]",str
"""AmazonEC2""","""us-east-1""","""BoxUsage:m5.large""",2160.0000000000,"""Hours""",207.3600000000,"""sku"""
"""AmazonEC2""","""us-east-1""","""BoxUsage:m5.xlarge""",720.0000000000,"""Hours""",0.0000000000,"""sku"""
"""AmazonEC2""","""us-east-1""","""EBS:VolumeUsage.gp3""",500.0000000000,"""GB-Months""",40.0000000000,"""sku"""
"""AmazonRDS""","""us-west-2""","""USW2-InstanceUsage:db.m5.large""",720.0000000000,"""Hours""",123.1200000000,"""sku"""
"""AmazonS3""","""us-west-2""","""USW2-TimedStorage-ByteHrs""",1840.0000000000,"""GB-Months""",42.3200000000,"""sku"""


In [11]:
print(len(frame.columns), "columns:", ", ".join(frame.columns))

34 columns: kind, provider, service_name, service_code, region_name, region_code, sku, usage_type, instance_type, operating_system, database_engine, deployment, storage_type, description, quantity, unit, cost, currency, unit_price, pricing_model, commitment_term, commitment_payment, period_start, period_end, account_id, account_name, resource_id, resource_type, extra, detail, source_file, source_page, source_row, item_id


`to_frame("full")` returns every stored column (75 of them, with names from the
[FOCUS](https://focus.finops.org) billing specification) for anything the short table leaves out.
`to_dicts()` gives plain nested dicts, ready for JSON:

In [12]:
import json

print(json.dumps(doc.commitments.to_dicts()[0], indent=1)[:900])

{
 "id": "sample-focus-export-csv-1d5b33dc:0:7",
 "kind": "commitment",
 "provider": "aws",
 "service": {
  "name": "Savings Plans for AWS Compute usage",
  "code": "ComputeSavingsPlans",
  "category": "Compute"
 },
 "region": {
  "name": null,
  "code": null,
  "zone": null
 },
 "product": {
  "sku": "SKU-SP-1Y-NU",
  "usage_type": "ComputeSP:1yrNoUpfront",
  "operation": null,
  "family": null,
  "meter": null,
  "instance_type": null,
  "operating_system": null,
  "database_engine": null,
  "deployment": null,
  "storage_type": null,
  "tenancy": null,
  "license": null,
  "extra": {}
 },
 "quantity": {
  "amount": "720",
  "unit": "Hours"
 },
 "cost": {
  "amount": "108",
  "currency": "USD"
 },
 "list_cost": {
  "amount": "108",
  "currency": "USD"
 },
 "unit_price": {
  "amount": "0.15",
  "currency": "USD"
 },
 "pricing": {
  "model": "committed_use",
  "term": null,
  "payment": 


## Infrastructure files

A Terraform or CloudFormation file has no costs, only resources. Its document has
`kind == "infrastructure"`, and `doc.usage` holds one item per billable resource, with the count
in `quantity`. This needs a model, so it is shown here without output:

```python
result = await parse("infra.zip")
for doc in result:
    for item in doc.usage:
        print(item.resource.type, item.resource.name, item.product.instance_type, item.quantity)

# aws_instance  web  m5.large  3
# aws_db_instance  primary  db.m5.large  None
```

`item.inferred` lists the fields that were worked out from elsewhere in the file (a variable's
default, say) and not written on the resource itself.

## Choosing a model

PDFs, spreadsheets and anything else without a recognised format are read with a language
model. The parser asks the model what things mean and then checks every number against the
source itself. Which model answers is one setting:

```python
await parse("bill.pdf", provider="claude")                      # ANTHROPIC_API_KEY
await parse("bill.pdf", provider="openai", model="<name>")      # OPENAI_API_KEY
await parse("bill.pdf", provider="vllm", base_url="https://…")  # a model you host
```

or `INVOICE_PARSER_PROVIDER` and `INVOICE_PARSER_MODEL` in the environment. A hosted provider
receives the text of the files you parse; a model you host does not leave your infrastructure.

For many files, keep one `Parser` and reuse it:

```python
from invoice_parser import Parser

parser = Parser(provider="claude", cache="./answers")   # never pay for the same question twice
results = await parser.parse_many(["a.pdf", "b.csv"])
```

## Field reference

| `Item` field | Type | Notes |
|---|---|---|
| `kind` | str | `usage`, `resource`, `commitment`, `discount`, `credit`, `tax`, `fee`, `adjustment` |
| `provider` | str | `aws`, `azure`, `gcp`, … |
| `service` | `Service(name, code, category)` | |
| `region` | `Region(name, code, zone)` | |
| `product` | `Product(sku, usage_type, operation, family, meter, instance_type, operating_system, database_engine, deployment, storage_type, tenancy, license, extra)` | `extra` holds anything else the document said about the product |
| `quantity` | `Quantity(amount, unit)` | |
| `cost` | `Money(amount, currency)` | What was charged |
| `list_cost`, `unit_price` | `Money` | When stated |
| `pricing` | `Pricing(model, term, payment, offering_class, commitment_id, commitment_type)` | `model`: `on_demand`, `reserved`, `savings_plan`, `spot`, `committed_use`, `free_tier`, `other` |
| `period` | `Period(start, end)` | |
| `account` | `Account(id, name)` | The linked or sub account when there is one |
| `resource` | `Resource(id, name, type, tags)` | |
| `description` | str | The line as the document words it |
| `detail` | str | See the table above |
| `source` | `Source(file, member, page, row, text)` | Where it was read from |
| `inferred` | tuple of str | Stored fields worked out from context |

All of these are immutable. Amounts are `Decimal`.